# 01 · What the compiler does with a fast and a slow config

Takes the fastest and the slowest valid config on the 4096³ matmul from the sweep, compiles both,
and keeps every stage of the lowering (TTIR, TTGIR, LLVM IR, PTX, SASS) side by side.

The slow config is a 256×256 tile on 2 warps: each thread has to hold far more accumulator values
than the 255-register limit allows. The PTX looks harmless, because PTX registers are virtual, but
after ptxas allocates real registers the SASS is full of `LDL`/`STL` spill traffic. The fast config
uses tensor cores (`mma.sync` → `HMMA`), `ldmatrix`, and on Ampere `cp.async` to pipeline its loads
through shared memory.

In [ ]:
# Setup: mount Drive, make sure the pinned torch/triton are installed, import archtune.
import importlib.metadata as md, os, subprocess, sys
PROJECT = "/content/drive/MyDrive/archtune"
NEED_GPU_STACK = True

if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

def _ver(pkg):
    try:
        return md.version(pkg).split("+")[0]
    except md.PackageNotFoundError:
        return None

if NEED_GPU_STACK and (_ver("torch"), _ver("triton")) != ("2.6.0", "3.2.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "triton==3.2.0"], check=True)
    if "torch" in sys.modules:
        raise SystemExit("torch was already imported: restart the session and run this cell again")

from archtune import config, gpu, model, compiler, run
ENV = gpu.capture()
MINI = config.mini_mode()
RESULTS = os.environ.get("ARCHTUNE_RESULTS") or os.path.join(PROJECT, "results_mini" if MINI else "results")
print(ENV["gpu_name"], "| torch", ENV["torch_version"], "| triton", ENV["triton_version"], "|", RESULTS)
if NEED_GPU_STACK and not gpu.pinned_ok(ENV):
    raise SystemExit("wrong torch/triton loaded: restart the session and run this cell again")

In [ ]:
DIMS = model.derive_dims(model.load_model_config(cache_dir=RESULTS))
MM_SHAPES = model.matmul_shapes(DIMS)
RMS_SHAPES = model.rmsnorm_shapes(DIMS)
SPLIT = {**model.train_test_split([s.shape_id for s in MM_SHAPES]),
         **model.train_test_split([s.shape_id for s in RMS_SHAPES])}
if MINI:
    small = model.mini_shapes(MM_SHAPES + RMS_SHAPES)
    MM_SHAPES = [s for s in small if s.kernel == "matmul"]
    RMS_SHAPES = [s for s in small if s.kernel == "rmsnorm"]
print(DIMS)
for s in MM_SHAPES + RMS_SHAPES:
    print(f"{s.shape_id:24s} {SPLIT[s.shape_id]:5s} {','.join(s.roles)}")

In [ ]:
import os, pandas as pd
mm = gpu.MATMUL
GPU_DIR = os.path.join(RESULTS, ENV["gpu_tag"])
SHAPE = next(s for s in MM_SHAPES if s.shape_id == config.EXPERIMENTS.large_shape_id)
sw = model.prepare(pd.read_parquet(os.path.join(GPU_DIR, "sweep.parquet")))
t = sw[(sw.shape_id == SHAPE.shape_id) & sw.valid].sort_values("time_ms_median")
FAST, SLOW = model.row_config(t.iloc[0]), model.row_config(t.iloc[-1])
print(f"fast {t.iloc[0].time_ms_median:.2f} ms", FAST)
print(f"slow {t.iloc[-1].time_ms_median:.2f} ms", SLOW)

In [ ]:
gpu.fresh_cache_dir()
inputs = mm.make_inputs(SHAPE.M, SHAPE.N, SHAPE.K)
LISTINGS, RES = {}, {}
for name, cfg in (("fast", FAST), ("slow", SLOW)):
    r = gpu.compile_kernel(mm, inputs, cfg)
    sass = gpu.disassemble(r.asm.get("cubin")) if r.asm.get("cubin") else None
    compiler.dump_ir(r.asm, os.path.join(GPU_DIR, "ir_exploration"), name, sass)
    LISTINGS[name] = {"ptx": r.asm.get("ptx"), "sass": sass}; RES[name] = r
    print(name, "regs", r.n_regs, "spills", r.n_spills, "smem", r.shared_bytes)
compiler.compare_features(LISTINGS)

## TTGIR layouts: how each tile is spread over threads and tensor cores

In [ ]:
for name in ("fast", "slow"):
    print(f"--- {name}"); print("\n".join(compiler.ttgir_layouts(RES[name].asm.get("ttgir", ""))))

## The instructions that matter

In [ ]:
for name in ("fast", "slow"):
    ptx = LISTINGS[name]["ptx"]
    for pat in (r"mma\.sync", r"ldmatrix", r"cp\.async\.c[ag]", r"(ld|st)\.local"):
        print(f"=== {name} PTX: {pat}\n{compiler.snippets(ptx, pat, context=0, max_hits=2)}\n")
    if LISTINGS[name]["sass"]:
        print(f"=== {name} SASS: spills\n{compiler.snippets(LISTINGS[name]['sass'], r'LDL|STL', context=0, max_hits=3)}\n")